<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/LogisticRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Quinto código - Cáncer de Mama 🌸

**Implementación de Logistic Regression para Clasificación de Cáncer de Mama**

###Importar librerías necesarias 🌺

In [ ]:
# ============================================
# CÓDIGO 5: LOGISTIC REGRESSION Y EVALUACIÓN
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🔬 CÓDIGO 5: EVALUACIÓN DE LOGISTIC REGRESSION")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔬 CÓDIGO 5: EVALUACIÓN DE LOGISTIC REGRESSION


###Cargar datos 🌺

In [ ]:
# 1. CARGAR DATOS
# ================
print(f"\n📂 Cargando datos del Código 2...")

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
    subsets = pickle.load(f)

y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')
y_test = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_final.npy')

print(f"✅ Datos cargados:")
print(f"   Subconjuntos: {list(subsets.keys())}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")


📂 Cargando datos del Código 2...
✅ Datos cargados:
   Subconjuntos: ['EO', 'BBA', 'CSA', 'RDA', 'GA']
   y_train: (206,)
   y_test: (28,)


###Configuración de Logistic Regression 🌺

In [ ]:
# 2. CONFIGURAR LOGISTIC REGRESSION (según Tabla 4 del artículo)
# ==============================================================
# El artículo usa: solver='lbfgs', penalty='l2', C=1.0
# Estos son los parámetros estándar que coinciden con lo que reportan

lr_params = {
    'solver': 'lbfgs',          # Algoritmo de optimización
    'penalty': 'l2',            # Regularización L2
    'C': 1.0,                   # Regularization strength (inverso de lambda)
    'max_iter': 1000,           # Máximo de iteraciones (garantiza convergencia)
    'random_state': 42,
    'class_weight': 'balanced'  # Maneja el desbalance de clases (aunque test está balanceado)
}

lr_model = LogisticRegression(**lr_params)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"\n⚙️ Configuración de Logistic Regression:")
for key, value in lr_params.items():
    print(f"   {key}: {value}")


⚙️ Configuración de Logistic Regression:
   solver: lbfgs
   penalty: l2
   C: 1.0
   max_iter: 1000
   random_state: 42
   class_weight: balanced


###Evaluación de cada subconjunto 🌺

In [ ]:
# 3. EVALUAR CADA SUBCONJUNTO
# ============================
results = {}

for name in subsets.keys():
    print(f"\n{'─'*40}")
    print(f"🔍 Subconjunto: {name} - Genes: {subsets[name]}")

    X_train_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy')
    X_test_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/test_{name}.npy')

    # Validación cruzada (10-fold)
    cv_acc = cross_val_score(lr_model, X_train_subset, y_train, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(lr_model, X_train_subset, y_train, cv=cv, scoring='f1')
    cv_auc = cross_val_score(lr_model, X_train_subset, y_train, cv=cv, scoring='roc_auc')

    # Entrenar modelo completo
    lr_model.fit(X_train_subset, y_train)

    # Predicciones
    y_pred = lr_model.predict(X_test_subset)
    y_proba = lr_model.predict_proba(X_test_subset)[:, 1]

    # Métricas en test
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_proba)

    # Coeficientes del modelo (para análisis de importancia de genes)
    # En regresión logística, los coeficientes indican el impacto de cada gen
    coefficients = pd.DataFrame({
        'gene': subsets[name],
        'coefficient': lr_model.coef_[0],
        'abs_coefficient': np.abs(lr_model.coef_[0])
    }).sort_values('abs_coefficient', ascending=False)

    results[name] = {
        'cv_accuracy': f"{cv_acc.mean():.4f} ± {cv_acc.std():.4f}",
        'cv_f1': f"{cv_f1.mean():.4f} ± {cv_f1.std():.4f}",
        'cv_auc': f"{cv_auc.mean():.4f} ± {cv_auc.std():.4f}",
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'test_auc': test_auc,
        'genes': subsets[name],
        'coefficients': coefficients,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'intercept': lr_model.intercept_[0]  # Término de bias (b0)
    }

    print(f"\n   Resultados:")
    print(f"   📊 Validación Cruzada (10-fold):")
    print(f"      Accuracy: {results[name]['cv_accuracy']}")
    print(f"      F1-Score: {results[name]['cv_f1']}")
    print(f"      AUC: {results[name]['cv_auc']}")
    print(f"\n   📈 Test:")
    print(f"      Accuracy: {test_acc:.4f}")
    print(f"      F1-Score: {test_f1:.4f}")
    print(f"      AUC: {test_auc:.4f}")
    print(f"      Intercept (b0): {lr_model.intercept_[0]:.4f}")
    print(f"\n   🔬 Coeficientes (importancia de genes):")
    print(coefficients[['gene', 'coefficient']].to_string(index=False))


────────────────────────────────────────
🔍 Subconjunto: EO - Genes: ['NM_002644', 'BM511013', 'NM_182611']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 1.0000 ± 0.0000
      F1-Score: 1.0000 ± 0.0000
      AUC: 1.0000 ± 0.0000

   📈 Test:
      Accuracy: 1.0000
      F1-Score: 1.0000
      AUC: 1.0000
      Intercept (b0): 25.8629

   🔬 Coeficientes (importancia de genes):
     gene  coefficient
NM_182611    -1.286876
 BM511013    -1.218575
NM_002644    -1.180789

────────────────────────────────────────
🔍 Subconjunto: BBA - Genes: ['NM_002644', 'BM511013']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 0.9950 ± 0.0150
      F1-Score: 0.9947 ± 0.0158
      AUC: 1.0000 ± 0.0000

   📈 Test:
      Accuracy: 0.9643
      F1-Score: 0.9811
      AUC: 1.0000
      Intercept (b0): 24.1742

   🔬 Coeficientes (importancia de genes):
     gene  coefficient
 BM511013    -1.834693
NM_002644    -1.552983

────────────────────────────────────────
🔍 Subconjun

###Mostrar resultados 🌺

In [ ]:
# 4. MOSTRAR RESULTADOS COMPARATIVOS
# ===================================
print(f"\n{'='*60}")
print("📊 RESULTADOS FINALES - LOGISTIC REGRESSION")
print('='*60)

comparison_df = pd.DataFrame({
    'Subset': results.keys(),
    'Genes': [', '.join(r['genes']) for r in results.values()],
    'Test_Accuracy': [f"{r['test_accuracy']:.4f}" for r in results.values()],
    'Test_F1': [f"{r['test_f1']:.4f}" for r in results.values()],
    'Test_AUC': [f"{r['test_auc']:.4f}" for r in results.values()],
    'CV_Accuracy': [r['cv_accuracy'] for r in results.values()]
})

print("\n📋 Tabla comparativa:")
print(comparison_df.to_string(index=False))


📊 RESULTADOS FINALES - LOGISTIC REGRESSION

📋 Tabla comparativa:
Subset                          Genes Test_Accuracy Test_F1 Test_AUC     CV_Accuracy
    EO NM_002644, BM511013, NM_182611        1.0000  1.0000   1.0000 1.0000 ± 0.0000
   BBA            NM_002644, BM511013        0.9643  0.9811   1.0000 0.9950 ± 0.0150
   CSA            BM511013, NM_182611        0.9643  0.9804   1.0000 1.0000 ± 0.0000
   RDA                      NM_002644        0.8571  0.9231   0.8462 0.9902 ± 0.0195
    GA                       AW015426        0.9643  0.9804   1.0000 0.9902 ± 0.0195
